# BLIP Fine-Tuning for Text-to-Video Retrieval

This notebook integrates BLIP with the project MSVD video-text format, uses the standardized three-frame configuration, fine-tunes BLIP with an image-text contrastive objective, and evaluates text-to-video retrieval.

`SMOKE_TEST` intentionally remains `True` for a safe reproducibility run. Set it to `False` only when performing the full experiment.

In [1]:
!pip install -q transformers accelerate decord pillow tqdm pandas nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 44.1 MB/s eta 0:00:00


In [2]:
import os, json, random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForImageTextRetrieval
from decord import VideoReader, cpu

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; using local paths.')


Mounted at /content/drive


In [3]:
# Reproducible experiment configuration
SEED = 298
DATA_ROOT = '/content/drive/Shared drives/DATA 298A/DATA/MSVD'
OUTPUT_DIR = '/content/blip_finetuned_outputs'
MODEL_NAME = 'Salesforce/blip-itm-base-coco'

# Shared project preprocessing configuration
FRAME_COUNT = 3
FRAME_SAMPLING = 'begin/mid/end'
IMAGE_SIZE = 224
FRAME_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'frames_224_jpeg')
TOP_K_FRAMES = 2
MAX_TEXT_LEN = 64

# Keep this True for the requested smoke test.
SMOKE_TEST = True
SMOKE_VIDEOS = 8
EPOCHS = 1
TRAIN_BATCH_SIZE = 4
EVAL_TEXT_BATCH_SIZE = 64
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_WORKERS = 0
# Evaluate all captions so BLIP uses the same query population as CLIP.
QUERY_POLICY = 'all captions per video'

os.makedirs(OUTPUT_DIR, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('SMOKE_TEST:', SMOKE_TEST)


Device: cuda
SMOKE_TEST: True


## 1. Shared MSVD loader and standardized frame sampling

The loader follows the project shared split schema: `msvd_train.json`, `msvd_val.json`, and `msvd_test.json`, with videos stored in `raw_videos`. It groups captions by video ID so the train/validation/test split remains video-disjoint. For each video, the preprocessing step extracts the beginning, middle, and end frames, resizes them to 224x224, and saves them as RGB JPEG files for BLIP. If the team has a Python shared-loader module, replace the body of `load_shared_split` with that project import; the remainder of the notebook remains unchanged.

In [4]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def clean_caption(text):
    return ' '.join(str(text).strip().split())

def make_video_index(video_root):
    index = {}
    for path in Path(video_root).iterdir():
        if path.is_file():
            index[path.name] = str(path)
            index[path.stem] = str(path)
    return index

def load_shared_split(split, max_videos=None):
    annotation_path = os.path.join(DATA_ROOT, f'msvd_{split}.json')
    video_root = os.path.join(DATA_ROOT, 'raw_videos')
    if not os.path.exists(annotation_path):
        raise FileNotFoundError(annotation_path)
    if not os.path.isdir(video_root):
        raise FileNotFoundError(video_root)

    video_index = make_video_index(video_root)
    by_video = {}
    for item in load_json(annotation_path):
        video_name = str(item.get('video', item.get('video_name', '')))
        video_path = video_index.get(video_name) or video_index.get(Path(video_name).stem)
        if not video_path:
            continue
        video_id = str(item.get('video_id', Path(video_name).stem))
        raw = item.get('caption', item.get('captions', []))
        captions = [raw] if isinstance(raw, str) else raw
        captions = [clean_caption(x) for x in captions if clean_caption(x)]
        if video_id not in by_video:
            by_video[video_id] = {'video_id': video_id, 'video_name': video_name, 'video_path': video_path, 'captions': []}
        by_video[video_id]['captions'].extend(captions)

    videos = list(by_video.values())
    for video in videos:
        video['captions'] = list(dict.fromkeys(video['captions']))
    videos.sort(key=lambda x: x['video_id'])
    return videos[:max_videos] if max_videos else videos

def sample_indices(total_frames, frame_count=FRAME_COUNT):
    if total_frames <= 0:
        raise ValueError('Video has no frames')
    if frame_count != 3:
        raise ValueError('This preprocessing configuration requires exactly 3 frames.')
    # Select the first, middle, and final frame to cover the whole video.
    return [0, total_frames // 2, total_frames - 1]

def extract_frames_as_jpegs(video, output_root=FRAME_OUTPUT_DIR):
    """Extract begin/mid/end frames and save BLIP-ready 224x224 JPEGs."""
    reader = VideoReader(video['video_path'], ctx=cpu(0))
    indices = sample_indices(len(reader), FRAME_COUNT)
    frame_names = ['begin', 'mid', 'end']
    video_dir = Path(output_root) / str(video['video_id'])
    video_dir.mkdir(parents=True, exist_ok=True)
    frame_paths = []

    # Save each selected frame as an RGB JPEG resized to BLIP's 224x224 input size.
    for frame_name, frame_array in zip(frame_names, reader.get_batch(indices).asnumpy()):
        image = Image.fromarray(frame_array).convert('RGB')
        image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
        frame_path = video_dir / f'{frame_name}.jpg'
        image.save(frame_path, format='JPEG', quality=95)
        frame_paths.append(str(frame_path))
    return frame_paths

def read_sampled_frames(video_path, frame_paths=None):
    # Read the saved JPEGs when available so training and evaluation use the
    # same explicit 224x224 begin/mid/end preprocessing.
    if frame_paths is not None:
        return [Image.open(path).convert('RGB') for path in frame_paths]
    reader = VideoReader(video_path, ctx=cpu(0))
    indices = sample_indices(len(reader), FRAME_COUNT)
    return [Image.fromarray(x).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
            for x in reader.get_batch(indices).asnumpy()]

limit = SMOKE_VIDEOS if SMOKE_TEST else None
train_videos = load_shared_split('train', limit)
val_videos = load_shared_split('val', limit)
test_videos = load_shared_split('test', limit)

# Pre-extract exactly three JPEG frames for every video in each split.
# The saved files make the preprocessing reproducible and easy to inspect.
for video in train_videos + val_videos + test_videos:
    video['frame_paths'] = extract_frames_as_jpegs(video)
print({'train': len(train_videos), 'val': len(val_videos), 'test': len(test_videos)})
assert set(x['video_id'] for x in train_videos).isdisjoint(x['video_id'] for x in val_videos)
assert set(x['video_id'] for x in train_videos).isdisjoint(x['video_id'] for x in test_videos)
assert set(x['video_id'] for x in val_videos).isdisjoint(x['video_id'] for x in test_videos)


{'train': 8, 'val': 8, 'test': 8}


## 2. BLIP-compatible dataset and smoke test

Each training item contains one standardized video frame and one caption. The smoke test confirms that the processor produces valid `pixel_values`, `input_ids`, and `attention_mask` tensors.

In [5]:
class FrameCaptionDataset(Dataset):
    def __init__(self, videos, processor, max_items=None):
        self.processor = processor
        self.items = []
        for video in videos:
            for caption in video['captions']:
                for frame_index in range(FRAME_COUNT):
                    self.items.append((video, caption, frame_index))
        if max_items is not None:
            self.items = self.items[:max_items]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        video, caption, frame_index = self.items[index]
        frame = read_sampled_frames(video['video_path'], video['frame_paths'])[frame_index]
        encoded = self.processor(images=frame, text=caption, padding='max_length', truncation=True, max_length=MAX_TEXT_LEN, return_tensors='pt')
        return {key: value.squeeze(0) for key, value in encoded.items()}

processor = BlipProcessor.from_pretrained(MODEL_NAME)
smoke_dataset = FrameCaptionDataset(train_videos, processor, max_items=min(4, len(train_videos) * FRAME_COUNT))
smoke_loader = DataLoader(smoke_dataset, batch_size=min(2, len(smoke_dataset)), num_workers=NUM_WORKERS)
smoke_batch = next(iter(smoke_loader))
print({key: tuple(value.shape) for key, value in smoke_batch.items()})
assert smoke_batch['pixel_values'].ndim == 4
assert smoke_batch['input_ids'].ndim == 2
assert smoke_batch['input_ids'].shape[-1] == MAX_TEXT_LEN
assert 'attention_mask' in smoke_batch
print('SMOKE TEST PASSED: BLIP-compatible tensors were created.')


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

{'input_ids': (2, 64), 'attention_mask': (2, 64), 'pixel_values': (2, 3, 384, 384)}
SMOKE TEST PASSED: BLIP-compatible tensors were created.


## 3. Fine-tune BLIP

The model is trained with a symmetric in-batch image-text contrastive loss. Checkpoints and training history are saved after every epoch.

In [6]:
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(DEVICE)
train_dataset = FrameCaptionDataset(train_videos, processor, max_items=32 if SMOKE_TEST else None)
val_dataset = FrameCaptionDataset(val_videos, processor, max_items=16 if SMOKE_TEST else None)
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

def move_batch(batch):
    return {key: value.to(DEVICE) for key, value in batch.items() if key in {'pixel_values', 'input_ids', 'attention_mask'}}

def contrastive_loss(batch):
    image_out = model.vision_model(pixel_values=batch['pixel_values'], return_dict=True)
    image_features = F.normalize(model.vision_proj(image_out.last_hidden_state[:, 0, :]), dim=-1)
    text_out = model.text_encoder(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], return_dict=True)
    text_features = F.normalize(model.text_proj(text_out.last_hidden_state[:, 0, :]), dim=-1)
    logits = image_features @ text_features.T
    labels = torch.arange(logits.size(0), device=DEVICE)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))

def run_epoch(loader, training):
    model.train(training)
    total, count = 0.0, 0
    for batch in tqdm(loader, leave=False, desc='train' if training else 'validation'):
        batch = move_batch(batch)
        with torch.set_grad_enabled(training):
            loss = contrastive_loss(batch)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        total += loss.item() * batch['input_ids'].size(0)
        count += batch['input_ids'].size(0)
    return total / max(count, 1)

history, best_val = [], float('inf')
for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, True)
    val_loss = run_epoch(val_loader, False)
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss}
    history.append(row)
    print(row)
    epoch_dir = os.path.join(OUTPUT_DIR, f'checkpoint-epoch-{epoch}')
    model.save_pretrained(epoch_dir)
    processor.save_pretrained(epoch_dir)
    if val_loss < best_val:
        best_val = val_loss
        model.save_pretrained(os.path.join(OUTPUT_DIR, 'best_checkpoint'))
        processor.save_pretrained(os.path.join(OUTPUT_DIR, 'best_checkpoint'))

with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)
print('Saved training evidence to', OUTPUT_DIR)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  895MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  895MB            

model.safetensors: downloading bytes:           |  0.00B            

...

validation:   0%|          | 0/4 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 1.3891021758317947, 'val_loss': 1.3865661025047302}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved training evidence to /content/blip_finetuned_outputs


## 4. Retrieval evaluation

Every available caption is evaluated as a separate text query. The caption's associated video is its ground-truth target, matching the CLIP evaluation protocol. The video score is the mean of the two strongest frame similarities.

In [7]:
@torch.inference_mode()
def encode_image_frames(model, processor, images):
    inputs = processor(images=images, return_tensors='pt')
    outputs = model.vision_model(pixel_values=inputs['pixel_values'].to(DEVICE), return_dict=True)
    features = model.vision_proj(outputs.last_hidden_state[:, 0, :])
    return F.normalize(features, dim=-1).cpu()

@torch.inference_mode()
def encode_text_queries(model, processor, texts):
    inputs = processor(text=texts, padding=True, truncation=True, max_length=MAX_TEXT_LEN, return_tensors='pt')
    outputs = model.text_encoder(input_ids=inputs['input_ids'].to(DEVICE), attention_mask=inputs['attention_mask'].to(DEVICE), return_dict=True)
    features = model.text_proj(outputs.last_hidden_state[:, 0, :])
    return F.normalize(features, dim=-1).cpu()

@torch.inference_mode()
def evaluate_retrieval(model, processor, videos):
    model.eval()
    video_frames, video_ids, queries = [], [], []
    for video in tqdm(videos, desc='Encoding videos'):
        video_frames.append(encode_image_frames(model, processor, read_sampled_frames(video['video_path'], video['frame_paths'])))
        video_ids.append(video['video_id'])
        # Add every caption so BLIP and CLIP use the same evaluation population.
        # A full MSVD test run should contain roughly 27,000 queries, rather
        # than only one query for each of approximately 670 videos.
        for caption in video['captions']:
            queries.append({'video_id': video['video_id'], 'caption': caption})

    text_chunks = []
    for start in range(0, len(queries), EVAL_TEXT_BATCH_SIZE):
        text_chunks.append(encode_text_queries(model, processor, [q['caption'] for q in queries[start:start + EVAL_TEXT_BATCH_SIZE]]))
    text_embeddings = torch.cat(text_chunks)

    scores = np.zeros((len(queries), len(videos)), dtype=np.float32)
    for video_index, frames in enumerate(video_frames):
        frame_scores = text_embeddings @ frames.T
        scores[:, video_index] = torch.topk(frame_scores, k=min(TOP_K_FRAMES, frames.shape[0]), dim=1).values.mean(dim=1).numpy()

    id_to_index = {video_id: index for index, video_id in enumerate(video_ids)}
    ranks = []
    for query_index, query in enumerate(queries):
        ranking = np.argsort(-scores[query_index])
        ranks.append(int(np.where(ranking == id_to_index[query['video_id']])[0][0]) + 1)
    ranks = np.asarray(ranks)
    return {
        'R@1': float(np.mean(ranks <= 1)),
        'R@5': float(np.mean(ranks <= 5)),
        'R@10': float(np.mean(ranks <= 10)),
        'MRR': float(np.mean(1.0 / ranks)),
        'MeanRank': float(np.mean(ranks)),
        'MedianRank': float(np.median(ranks)),
        'num_queries': int(len(ranks)),
        'num_videos': int(len(videos))
    }

best_model = BlipForImageTextRetrieval.from_pretrained(os.path.join(OUTPUT_DIR, 'best_checkpoint')).to(DEVICE)
best_processor = BlipProcessor.from_pretrained(os.path.join(OUTPUT_DIR, 'best_checkpoint'))
fine_tuned_metrics = evaluate_retrieval(best_model, best_processor, test_videos)
print(pd.DataFrame([fine_tuned_metrics]))
with open(os.path.join(OUTPUT_DIR, 'fine_tuned_blip_metrics.json'), 'w') as f:
    json.dump(fine_tuned_metrics, f, indent=2)


Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

Encoding videos:   0%|          | 0/8 [00:00<?, ?it/s]

        R@1       R@5  R@10       MRR  MeanRank  MedianRank  num_queries  \
0  0.706271  0.990099   1.0  0.814474   1.60396         1.0          303   

   num_videos  
0           8  


## 5. Zero-shot comparison

The zero-shot model uses the same test videos, frame sampling, pooling, query policy, and metrics as the fine-tuned model.

In [8]:
zero_shot_model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(DEVICE)
zero_shot_metrics = evaluate_retrieval(zero_shot_model, processor, test_videos)
with open(os.path.join(OUTPUT_DIR, 'zero_shot_blip_metrics.json'), 'w') as f:
    json.dump(zero_shot_metrics, f, indent=2)

metric_names = ['R@1', 'R@5', 'R@10', 'MRR', 'MeanRank', 'MedianRank']
comparison = pd.DataFrame({
    'Metric': metric_names,
    'Zero-shot BLIP': [zero_shot_metrics[x] for x in metric_names],
    'Fine-tuned BLIP': [fine_tuned_metrics[x] for x in metric_names]
})
comparison['Change'] = comparison['Fine-tuned BLIP'] - comparison['Zero-shot BLIP']
display(comparison)

configuration = {
    'dataset': 'MSVD',
    'model': MODEL_NAME,
    'frame_count': FRAME_COUNT,
    'frame_sampling': FRAME_SAMPLING,
    'image_size': IMAGE_SIZE,
    'top_k_frames': TOP_K_FRAMES,
    'query_policy': QUERY_POLICY,
    'smoke_test': SMOKE_TEST,
    'smoke_videos': SMOKE_VIDEOS,
    'epochs': EPOCHS,
    'batch_size': TRAIN_BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'max_text_length': MAX_TEXT_LEN,
    'seed': SEED
}
with open(os.path.join(OUTPUT_DIR, 'blip_zero_shot_vs_fine_tuned.json'), 'w') as f:
    json.dump({'zero_shot': zero_shot_metrics, 'fine_tuned': fine_tuned_metrics, 'configuration': configuration}, f, indent=2)


Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

Encoding videos:   0%|          | 0/8 [00:00<?, ?it/s]

,Metric,Zero-shot BLIP,Fine-tuned BLIP,Change
0,R@1,0.848185,0.706271,-0.141914
1,R@5,0.993399,0.990099,-0.003300
2,R@10,1.000000,1.000000,0.000000
3,MRR,0.903332,0.814474,-0.088857
4,MeanRank,1.336634,1.603960,0.267327
5,MedianRank,1.000000,1.000000,0.000000


## 6. Caption-generation metrics

The retrieval checkpoint above is a `BlipForImageTextRetrieval` model and cannot generate captions. This section therefore uses BLIP's pretrained captioning model to report BLEU and CIDEr against all available reference captions. These caption scores are separate from the fine-tuned retrieval R@K scores.


In [9]:
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from transformers import BlipForConditionalGeneration

CAPTION_MODEL_NAME = 'Salesforce/blip-image-captioning-base'
caption_processor = BlipProcessor.from_pretrained(CAPTION_MODEL_NAME)
caption_model = BlipForConditionalGeneration.from_pretrained(CAPTION_MODEL_NAME).to(DEVICE)
caption_model.eval()

def caption_ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))

def build_cider_document_frequency(videos):
    document_frequency = {n: Counter() for n in range(1, 5)}
    for video in videos:
        for n in range(1, 5):
            unique_ngrams = set()
            for reference in video['captions']:
                unique_ngrams.update(caption_ngrams(reference.lower().split(), n))
            for ngram in unique_ngrams:
                document_frequency[n][ngram] += 1
    return document_frequency

def cider_video_score(candidate, references, document_frequency, document_count):
    candidate_tokens = candidate.lower().split()
    reference_tokens = [reference.lower().split() for reference in references]
    scores = []
    for n in range(1, 5):
        candidate_counts = caption_ngrams(candidate_tokens, n)
        candidate_total = max(sum(candidate_counts.values()), 1)
        candidate_vector = {}
        for ngram, count in candidate_counts.items():
            idf = np.log(max(document_count, 1) / (1.0 + document_frequency[n][ngram]))
            candidate_vector[ngram] = (count / candidate_total) * idf
        reference_scores = []
        for reference in reference_tokens:
            reference_counts = caption_ngrams(reference, n)
            reference_total = max(sum(reference_counts.values()), 1)
            reference_vector = {}
            for ngram, count in reference_counts.items():
                idf = np.log(max(document_count, 1) / (1.0 + document_frequency[n][ngram]))
                reference_vector[ngram] = (count / reference_total) * idf
            common = set(candidate_vector) & set(reference_vector)
            numerator = sum(candidate_vector[x] * reference_vector[x] for x in common)
            candidate_norm = np.sqrt(sum(x * x for x in candidate_vector.values()))
            reference_norm = np.sqrt(sum(x * x for x in reference_vector.values()))
            similarity = numerator / (candidate_norm * reference_norm) if candidate_norm and reference_norm else 0.0
            length_penalty = np.exp(-((len(candidate_tokens) - len(reference)) ** 2) / (2 * 6.0 ** 2))
            reference_scores.append(similarity * length_penalty)
        scores.append(np.mean(reference_scores) if reference_scores else 0.0)
    return float(10.0 * np.mean(scores))

@torch.inference_mode()
def generate_video_caption(video):
    frames = read_sampled_frames(video['video_path'])
    image = frames[len(frames) // 2]
    inputs = caption_processor(images=image, return_tensors='pt').to(DEVICE)
    output = caption_model.generate(**inputs, max_new_tokens=30, num_beams=3)
    return caption_processor.decode(output[0], skip_special_tokens=True).strip()

document_frequency = build_cider_document_frequency(test_videos)
caption_predictions = []
smoothing = SmoothingFunction().method1
for video in tqdm(test_videos, desc='Generating captions'):
    prediction = generate_video_caption(video)
    references = video['captions']
    reference_tokens = [reference.lower().split() for reference in references]
    prediction_tokens = prediction.lower().split()
    caption_predictions.append({
        'video_id': video['video_id'],
        'prediction': prediction,
        'BLEU-1': sentence_bleu(reference_tokens, prediction_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothing),
        'BLEU-2': sentence_bleu(reference_tokens, prediction_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing),
        'BLEU-3': sentence_bleu(reference_tokens, prediction_tokens, weights=(1/3, 1/3, 1/3, 0), smoothing_function=smoothing),
        'BLEU-4': sentence_bleu(reference_tokens, prediction_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing),
        'CIDEr': cider_video_score(prediction, references, document_frequency, len(test_videos))
    })

caption_metrics = {
    metric: float(np.mean([row[metric] for row in caption_predictions]))
    for metric in ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'CIDEr']
}
caption_metrics['num_videos'] = len(caption_predictions)
display(pd.DataFrame([caption_metrics]))
with open(os.path.join(OUTPUT_DIR, 'blip_caption_metrics.json'), 'w') as f:
    json.dump({'metrics': caption_metrics, 'predictions': caption_predictions}, f, indent=2)


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Generating captions:   0%|          | 0/8 [00:00<?, ?it/s]

,BLEU-1,BLEU-2,BLEU-3,BLEU-4,CIDEr,num_videos
0,0.7459,0.520088,0.299347,0.19447,0.392709,8


## Evidence checklist

This smoke run should produce:

- `SMOKE TEST PASSED` output with tensor shapes
- `training_history.json`
- `checkpoint-epoch-1/` and `best_checkpoint/`
- `fine_tuned_blip_metrics.json`
- `zero_shot_blip_metrics.json`
- `blip_zero_shot_vs_fine_tuned.json`
- `blip_caption_metrics.json`

Because `SMOKE_TEST = True`, this run is integration evidence rather than final performance evidence. For final project metrics, rerun the same notebook with `SMOKE_TEST = False` and retain the resulting checkpoint and JSON files.